# Training the model

Part one of a two-notebook exercise on deployment. This notebook trains a cat-vs-dog
image classifier and exports it to `model.pkl`; [gradio.ipynb](./gradio.ipynb) then loads
that file and puts a browser interface in front of it.

**Run this notebook first.** `model.pkl` is build output, not source, so it is not checked
into the repository — `gradio.ipynb` cannot run until this notebook has produced it.

In [ ]:
!pip install -Uqq fastai

In [ ]:
from fastai.vision.all import *

## Get the data

The Oxford-IIIT Pet dataset, downloaded and cached by fastai. Unlike the bird exercise
there is no scraping to do — this is a curated dataset with a known structure.

In [ ]:
path = untar_data(URLs.PETS)/'images'

## Label the images

The dataset encodes its labels in the filenames: cat breeds are capitalised, dog breeds
are not. So the entire labelling function is a check on the first character.

In [ ]:
def is_cat(x): return x[0].isupper() 

In [ ]:
dls = ImageDataLoaders.from_name_func('.',
    get_image_files(path), valid_pct=0.2, seed=42,
    label_func=is_cat,
    item_tfms=Resize(192))

## Train

A pretrained `resnet18`, fine-tuned for three epochs, with `error_rate` reported on the
20% validation split.

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

## Export

`learn.export` writes the model architecture *and* its trained weights *and* the inference
pipeline to a single `.pkl`. That file is self-contained: the next notebook loads it in a
process that has never seen this dataset.

In [ ]:
learn.export('model.pkl')